# AutoClassify — Stage 2 (Colab, end to end)

**Reinforcement learning for budget-aware algorithm selection.** An RL agent decides, step by step, which classifier to evaluate at which training-data budget (10/25/50/100%) and when to stop, so it finds a strong classifier for a new dataset at a fraction of the cost of trying everything.

This notebook runs the whole of Stage 2 on **25 OpenML-CC18 datasets** and writes everything to your Google Drive:

1. downloads and cleans the datasets
2. builds the performance database (12 classifiers × 4 budgets × 3 seeds), **resumable**
3. computes dataset features for the agent's state
4. **5-fold cross-validation over datasets**: every dataset is used once as an unseen test dataset, so all 25 datasets count in the evaluation
5. trains tabular Q-learning, DQN and Double DQN (3 seeds each) and 9 baselines
6. runs ablations, statistical tests, and saves figures, tables and model weights

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**. If Colab disconnects, run all again: finished work is skipped.

## 1. Settings
Change `RUN_NAME` for a fresh run. Set `TEST_MODE = True` for a 5-minute check on synthetic data (no downloads).

In [ ]:
RUN_NAME = "stage2_v1"
DRIVE_ROOT = "/content/drive/MyDrive/AutoClassify"
TEST_MODE = False          # True = quick synthetic check of the whole notebook

# 25 OpenML-CC18 datasets (same as your Stage 2)
DATASET_IDS = [11, 14, 22, 28, 29, 31, 37, 50, 54, 188, 469, 1068, 1464, 1480, 1494,
               1510, 1590, 4534, 4538, 23517, 40975, 40979, 40983, 40984, 41027]
SEEDS = [0, 1, 2]                      # row-split seeds per dataset
BUDGETS = [0.1, 0.25, 0.5, 1.0]        # fractions of the training rows
CLASSIFIERS_WANTED = ["logreg", "lda", "gnb", "knn", "svm", "dtree", "rf", "extratrees",
                      "lightgbm", "catboost", "realmlp", "tabpfn"]
FALLBACK_POOL = ["hgb", "mlp_sklearn"]  # used only if an optional classifier fails to install/run
TIMEOUT_S = 300                         # per fit
MAX_ROWS = 50000
N_FOLDS = 5                             # cross-validation over datasets
AGENT_SEEDS = [0, 1, 2]
ENV_CFG = dict(lam=0.05,               # cost penalty per unit of budget
               obs_noise=0.01,         # noise on observed val scores, TRAINING episodes only
               shaping=True,           # potential-based reward shaping (Ng et al., 1999), training only
               cost_mask=True)         # same rule for every method: actions whose ESTIMATED cost exceeds the remaining budget are invalid
DQN_CFG = dict(hidden=[128, 128], lr=3e-4, gamma=0.99, batch_size=128, buffer_size=100000,
               warmup=2000, target_update=1000, train_steps=60000, eps_start=1.0, eps_end=0.05,
               eps_decay_frac=0.4, eval_every=5000, grad_clip=10.0)
QL_CFG = dict(episodes=40000, alpha=0.1, gamma=1.0, eps_start=1.0, eps_end=0.05,
              eps_decay_frac=0.5, eval_every=5000)
RUN_ABLATIONS = True
ABLATIONS = {"no_dataset_features": {"features": False},
             "no_observed_costs": {"cost": False},
             "no_budget_info": {"budget": False},
             "one_shot_no_observations": {"obs": False},
             "full_budget_only": "full_budget_only"}

if TEST_MODE:   # tiny settings for a quick end-to-end check
    RUN_NAME = RUN_NAME + "_TEST"
    DATASET_IDS = list(range(900000, 900015))
    SEEDS = [0, 1]
    CLASSIFIERS_WANTED = ["logreg", "lda", "gnb", "knn", "svm", "dtree", "rf", "extratrees", "hgb", "mlp_sklearn"]
    AGENT_SEEDS = [0, 1]
    DQN_CFG.update(train_steps=3000, warmup=300, eval_every=1500, target_update=300, buffer_size=10000)
    QL_CFG.update(episodes=3000, eval_every=1500)

## 2. Google Drive, GPU check and installs

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    DRIVE_ROOT = os.path.abspath("./AutoClassify_local")

OUT = os.path.join(DRIVE_ROOT, RUN_NAME)
DIRS = {k: os.path.join(OUT, k) for k in
        ["data", "perfdb", "features", "agents", "results", "figures", "tables", "openml_cache"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
print("Everything is saved under:", OUT)

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except Exception:
    gpu = ""
print("GPU:", gpu or "none (Runtime -> Change runtime type -> T4 GPU)")

In [ ]:
if IN_COLAB and not TEST_MODE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "catboost", "pytabkit",
                    "tabpfn", "gymnasium", "scipy", "pandas", "scikit-learn", "matplotlib"], check=False)
print("installs done. If Colab asks you to RESTART the session, do it (Runtime -> Restart session), then Run all again.")

## 3. Imports and helpers

In [ ]:
import json, math, time, warnings, pickle, re, inspect, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.tolist() if isinstance(o, np.ndarray) else float(o))

def load_json(path):
    with open(path) as f:
        return json.load(f)

def log(msg):
    print(time.strftime("%H:%M:%S"), "|", msg, flush=True)

def cuda_available():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

HARDWARE = f"{platform.processor() or 'cpu'} | cores={os.cpu_count()} | gpu={gpu or 'none'}"
N_JOBS = max(1, os.cpu_count() or 1)
DEVICE = "cuda" if cuda_available() else "cpu"
print("device for TabPFN/RealMLP:", DEVICE, "| CPU cores:", N_JOBS)

## 4. Candidate classifiers
Fixed default hyperparameters (this is algorithm selection, not hyperparameter tuning). Each optional classifier is test-fitted once; if it fails, it is replaced from the fallback pool **before** any real fit, so every dataset uses the same set.

In [ ]:
def _make(cls, **kw):
    try:
        params = inspect.signature(cls.__init__).parameters
        if any(p.kind == p.VAR_KEYWORD for p in params.values()):
            return cls(**kw)
        return cls(**{k: v for k, v in kw.items() if k in params})
    except (TypeError, ValueError):
        return cls()

def make_classifier(name, seed):
    if name == "logreg":
        from sklearn.linear_model import LogisticRegression; return LogisticRegression(max_iter=1000, random_state=seed)
    if name == "lda":
        from sklearn.discriminant_analysis import LinearDiscriminantAnalysis; return LinearDiscriminantAnalysis()
    if name == "gnb":
        from sklearn.naive_bayes import GaussianNB; return GaussianNB()
    if name == "knn":
        from sklearn.neighbors import KNeighborsClassifier; return KNeighborsClassifier(n_neighbors=5, n_jobs=N_JOBS)
    if name == "svm":
        from sklearn.svm import SVC; return SVC(kernel="rbf", random_state=seed, cache_size=500)
    if name == "dtree":
        from sklearn.tree import DecisionTreeClassifier; return DecisionTreeClassifier(random_state=seed)
    if name == "rf":
        from sklearn.ensemble import RandomForestClassifier; return RandomForestClassifier(n_estimators=100, n_jobs=N_JOBS, random_state=seed)
    if name == "extratrees":
        from sklearn.ensemble import ExtraTreesClassifier; return ExtraTreesClassifier(n_estimators=100, n_jobs=N_JOBS, random_state=seed)
    if name == "hgb":
        from sklearn.ensemble import HistGradientBoostingClassifier; return HistGradientBoostingClassifier(random_state=seed)
    if name == "mlp_sklearn":
        from sklearn.neural_network import MLPClassifier; return MLPClassifier(hidden_layer_sizes=(100,), early_stopping=True, max_iter=300, random_state=seed)
    if name == "lightgbm":
        from lightgbm import LGBMClassifier; return LGBMClassifier(n_jobs=N_JOBS, random_state=seed, verbose=-1)
    if name == "catboost":
        from catboost import CatBoostClassifier; return CatBoostClassifier(verbose=0, thread_count=N_JOBS, random_seed=seed, allow_writing_files=False)
    if name == "realmlp":
        from pytabkit import RealMLP_TD_Classifier; return _make(RealMLP_TD_Classifier, device=DEVICE, random_state=seed, n_threads=N_JOBS, verbosity=0)
    if name == "tabpfn":
        from tabpfn import TabPFNClassifier; return _make(TabPFNClassifier, device=DEVICE, random_state=seed, ignore_pretraining_limits=True)
    raise KeyError(name)

FAMILY = {"logreg": "linear", "lda": "linear (generative)", "gnb": "probabilistic", "knn": "instance-based",
          "svm": "kernel", "dtree": "single tree", "rf": "bagging", "extratrees": "randomized trees",
          "lightgbm": "boosting", "catboost": "boosting", "hgb": "boosting", "realmlp": "deep learning (MLP)",
          "tabpfn": "deep learning (transformer)", "mlp_sklearn": "neural network"}
RAW_CAT = {"catboost"}   # CatBoost gets raw categorical columns

## 5. Data: download, clean, filter

In [ ]:
def categorical_columns(X):
    return [c for c in X.columns if isinstance(X[c].dtype, pd.CategoricalDtype) or X[c].dtype == object
            or pd.api.types.is_bool_dtype(X[c]) or pd.api.types.is_string_dtype(X[c])]

def make_synthetic(i, seed=0):
    from sklearn.datasets import make_classification
    rng = np.random.default_rng(seed * 1000 + i)
    n, d, c = int(rng.integers(500, 3000)), int(rng.integers(4, 40)), int(rng.integers(2, 6))
    inf = max(2, int(d * rng.uniform(0.3, 0.9))); clusters = int(rng.integers(1, 3))
    while c * clusters > 2 ** inf:
        if inf < d: inf += 1
        else: clusters = 1; break
    X, y = make_classification(n_samples=n, n_features=d, n_informative=inf, n_redundant=0, n_classes=c,
                               n_clusters_per_class=clusters, class_sep=float(rng.uniform(0.4, 1.8)),
                               flip_y=float(rng.uniform(0, 0.08)), random_state=int(rng.integers(1e9)))
    X = pd.DataFrame(X, columns=[f"f{j}" for j in range(d)])
    if rng.random() < 0.5:
        X["f0"] = "c" + pd.cut(X["f0"], bins=5, labels=False).astype(str)
    return X, pd.Series(y.astype(str)), f"synthetic_{i}"

def clean(X, y, min_rows=500, max_rows=MAX_ROWS, max_enc=500, max_classes=50, min_class=10, max_cat=20):
    X = X.reset_index(drop=True).copy(); y = y.astype(str).reset_index(drop=True)
    keep = y.notna() & (y != "nan"); X, y = X[keep].reset_index(drop=True), y[keep].reset_index(drop=True)
    cats = categorical_columns(X)
    for c in X.columns:
        if c in cats:
            X[c] = X[c].astype(object).where(X[c].notna(), None).map(lambda v: None if v is None else str(v)).astype(object)
        else:
            X[c] = pd.to_numeric(X[c], errors="coerce").astype(float)
    drop = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
    drop += [c for c in cats if c not in drop and X[c].nunique() > 0.9 * len(X)]
    X = X.drop(columns=drop)
    vc = y.value_counts(); rare = vc[vc < min_class].index
    if len(rare):
        m = ~y.isin(rare); X, y = X[m].reset_index(drop=True), y[m].reset_index(drop=True)
    if y.nunique() < 2 or y.nunique() > max_classes or len(y) < min_rows or X.shape[1] == 0:
        raise ValueError("fails filters (classes/rows/features)")
    if len(y) > max_rows:
        idx, _ = train_test_split(np.arange(len(y)), train_size=max_rows, stratify=y, random_state=0)
        idx = np.sort(idx); X, y = X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
    cats = categorical_columns(X)
    n_enc = X.shape[1] - len(cats) + sum(min(X[c].nunique(), max_cat) for c in cats)
    if n_enc > max_enc:
        raise ValueError(f"{n_enc} encoded features > {max_enc}")
    return X, y

def fetch_all():
    meta_path = os.path.join(DIRS["data"], "metadata.csv")
    rows = []
    for did in DATASET_IDS:
        out = os.path.join(DIRS["data"], f"{did}.pkl")
        if os.path.exists(out):
            d = pd.read_pickle(out)
        else:
            try:
                if TEST_MODE:
                    X, y, name = make_synthetic(did - 900000)
                else:
                    from sklearn.datasets import fetch_openml
                    b = fetch_openml(data_id=did, as_frame=True, parser="auto", data_home=DIRS["openml_cache"])
                    X, y, name = b.data, b.target, b.details.get("name", str(did))
                X, y = clean(X, y)
                d = {"X": X, "y": y, "name": name, "did": did}
                pd.to_pickle(d, out)
            except Exception as e:
                log(f"[rejected] {did}: {e}"); continue
        X, y = d["X"], d["y"]
        rows.append({"did": did, "name": d["name"], "n_rows": len(y), "n_features": X.shape[1],
                     "n_categorical": len(categorical_columns(X)), "n_classes": y.nunique(),
                     "minority_share": round(y.value_counts(normalize=True).min(), 4)})
        log(f"[ok] {did} {d['name']}: {len(y)} rows, {X.shape[1]} features, {y.nunique()} classes")
    meta = pd.DataFrame(rows); meta.to_csv(meta_path, index=False)
    return meta

META = fetch_all()
DIDS = sorted(META["did"].tolist())
print(f"\n{len(DIDS)} datasets ready")
META

## 6. Splits and preprocessing
* **Rows** (inside each dataset and seed): stratified 60% train / 20% validation / 20% test.
* **Budgets**: nested stratified subsamples of the training rows (10% ⊂ 25% ⊂ 50% ⊂ 100%). Every fit is from scratch.
* **Preprocessing** is fitted only on the rows a model is trained on: median impute + scale numeric, one-hot categorical (CatBoost gets raw categoricals).

In [ ]:
def row_split(y, seed):
    idx = np.arange(len(y))
    tr, rest = train_test_split(idx, train_size=0.6, stratify=y, random_state=seed)
    va, te = train_test_split(rest, train_size=0.5, stratify=y.iloc[rest], random_state=seed + 10000)
    return np.sort(tr), np.sort(va), np.sort(te)

def budget_subsample(y_train, frac, seed):
    y_train = np.asarray(y_train); rng = np.random.default_rng(seed + 777); keep = []
    for c in np.unique(y_train):
        pos = np.flatnonzero(y_train == c); pos = pos[rng.permutation(len(pos))]
        keep.append(pos[:max(1, math.ceil(frac * len(pos)))])
    return np.sort(np.concatenate(keep))

class StandardPre:
    def fit(self, X):
        cats = categorical_columns(X); nums = [c for c in X.columns if c not in cats]; parts = []
        if nums: parts.append(("n", Pipeline([("i", SimpleImputer(strategy="median", keep_empty_features=True)), ("s", StandardScaler())]), nums))
        if cats: parts.append(("c", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="missing")),
                                             ("o", OneHotEncoder(handle_unknown="infrequent_if_exist", max_categories=20, sparse_output=False))]), cats))
        self.ct = ColumnTransformer(parts, sparse_threshold=0.0).fit(X); return self
    def transform(self, X):
        return np.nan_to_num(self.ct.transform(X).astype(np.float32))

class RawCatPre:
    def fit(self, X):
        self.cols = list(X.columns); self.cats = categorical_columns(X)
        self.cat_idx = [self.cols.index(c) for c in self.cats]; return self
    def transform(self, X):
        X = X[self.cols].copy()
        for c in self.cats: X[c] = X[c].astype(object).where(X[c].notna(), "missing").astype(str)
        return X

def fit_eval(name, Xtr, ytr, Xva, yva, Xte, yte, seed):
    t0 = time.perf_counter()
    pre = (RawCatPre() if name in RAW_CAT else StandardPre()).fit(Xtr)
    A = pre.transform(Xtr); m = make_classifier(name, seed)
    if name in RAW_CAT: m.fit(A, ytr, cat_features=pre.cat_idx)
    else: m.fit(A, ytr)
    t1 = time.perf_counter(); pva = np.asarray(m.predict(pre.transform(Xva))).ravel().astype(int)
    t2 = time.perf_counter(); pte = np.asarray(m.predict(pre.transform(Xte))).ravel().astype(int)
    return {"val_bacc": balanced_accuracy_score(yva, pva), "test_bacc": balanced_accuracy_score(yte, pte),
            "val_f1": f1_score(yva, pva, average="macro"), "test_f1": f1_score(yte, pte, average="macro"),
            "fit_time": t1 - t0, "predict_val_time": t2 - t1}

### Decide the classifier set (once per run, then frozen)

In [ ]:
def probe(name):
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=200, n_features=6, n_informative=4, n_classes=3, random_state=0)
    X = pd.DataFrame(X, columns=[f"f{i}" for i in range(6)])
    fit_eval(name, X.iloc[:120], y[:120], X.iloc[120:160], y[120:160], X.iloc[160:], y[160:], 0)  # also warms up GPU / downloads weights

cfg_path = os.path.join(OUT, "run_config.json")
if os.path.exists(cfg_path):
    CLASSIFIERS = load_json(cfg_path)["classifiers"]
    log(f"classifier set loaded from previous session (frozen): {CLASSIFIERS}")
    for c in CLASSIFIERS:
        probe(c)   # warm-up only
else:
    CLASSIFIERS, pool = [], list(FALLBACK_POOL)
    for c in CLASSIFIERS_WANTED:
        try:
            probe(c); CLASSIFIERS.append(c); log(f"  {c}: ok")
        except Exception as e:
            log(f"  {c}: FAILED ({type(e).__name__}: {str(e)[:120]})")
            while pool:
                alt = pool.pop(0)
                if alt in CLASSIFIERS or alt in CLASSIFIERS_WANTED: continue
                try:
                    probe(alt); CLASSIFIERS.append(alt); log(f"  -> using fallback {alt}"); break
                except Exception: pass
    save_json({"classifiers": CLASSIFIERS, "budgets": BUDGETS, "seeds": SEEDS, "datasets": DIDS,
               "hardware": HARDWARE, "env": ENV_CFG, "dqn": DQN_CFG, "qlearning": QL_CFG}, cfg_path)
K, L = len(CLASSIFIERS), len(BUDGETS)
print(f"\nUsing {K} classifiers:", CLASSIFIERS)

## 7. Performance database (the long step, resumable)
Every (dataset, seed, classifier, budget) is fitted once. **Cost** = seconds of preprocessing + fit + validation prediction (test prediction is not charged). A fit over `TIMEOUT_S` counts as a time-out (chance score, cost = timeout); larger budgets are skipped when the extrapolated time exceeds the limit. Each row is appended to Drive immediately.

In [ ]:
COLS = ["did", "seed", "clf", "level", "budget", "n_train", "status", "val_bacc", "test_bacc", "val_f1", "test_f1",
        "fit_time", "predict_val_time", "cost_s", "chance", "message"]
GROWTH = {"svm": 2.0, "knn": 1.0}

def run_dataset(did):
    path = os.path.join(DIRS["perfdb"], f"{did}.csv")
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=COLS)
    have = set(zip(done["seed"].astype(int), done["clf"], done["level"].astype(int)))
    todo = len(SEEDS) * K * L - len(have & {(s, c, l) for s in SEEDS for c in CLASSIFIERS for l in range(L)})
    if todo <= 0:
        return
    d = pd.read_pickle(os.path.join(DIRS["data"], f"{did}.pkl"))
    X, y = d["X"], LabelEncoder().fit_transform(d["y"]); chance = 1.0 / len(np.unique(y))
    log(f"[run] {did} {d['name']}: {todo} fits")
    new = []
    for seed in SEEDS:
        tr, va, te = row_split(pd.Series(y), seed)
        for c in CLASSIFIERS:
            prev_t, prev_f, dead = None, None, False
            for l, f in enumerate(BUDGETS):
                if (seed, c, l) in have:
                    r = done[(done.seed == seed) & (done.clf == c) & (done.level == l)].iloc[0]
                    dead = dead or str(r.status).startswith("timeout"); prev_t, prev_f = r.cost_s, f
                    continue
                pos = budget_subsample(y[tr], f, seed)
                base = dict(did=did, seed=seed, clf=c, level=l, budget=f, n_train=len(pos), chance=chance, message="")
                pred = prev_t * (f / prev_f) ** GROWTH.get(c, 1.2) if prev_t else 0.0
                if dead or pred > TIMEOUT_S:
                    row = {**base, "status": "timeout_predicted", "val_bacc": chance, "test_bacc": chance, "cost_s": TIMEOUT_S}
                    dead = True
                else:
                    t0 = time.perf_counter()
                    try:
                        r = fit_eval(c, X.iloc[tr[pos]], y[tr[pos]], X.iloc[va], y[va], X.iloc[te], y[te], seed)
                        t = r["fit_time"] + r["predict_val_time"]
                        if t > TIMEOUT_S:
                            row = {**base, **r, "status": "timeout", "val_bacc": chance, "test_bacc": chance, "cost_s": TIMEOUT_S}; dead = True
                        else:
                            row = {**base, **r, "status": "ok", "cost_s": t}
                        prev_t, prev_f = t, f
                    except Exception as e:
                        t = time.perf_counter() - t0
                        row = {**base, "status": "error", "val_bacc": chance, "test_bacc": chance, "cost_s": min(t, TIMEOUT_S),
                               "message": f"{type(e).__name__}: {str(e)[:150]}"}
                pd.DataFrame([row]).reindex(columns=COLS).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

t_start = time.time()
for i, did in enumerate(DIDS):
    run_dataset(did)
log(f"performance database complete ({time.time() - t_start:.0f}s this session)")

PERF = pd.concat([pd.read_csv(os.path.join(DIRS["perfdb"], f"{d}.csv")) for d in DIDS], ignore_index=True)
PERF = PERF[PERF["clf"].isin(CLASSIFIERS) & PERF["seed"].isin(SEEDS)]
print("rows:", len(PERF), "expected:", len(DIDS) * len(SEEDS) * K * L)
display_status = PERF.pivot_table(index="clf", columns="status", values="did", aggfunc="count", fill_value=0)
display_status["median_s_at_100%"] = PERF[PERF.level == L - 1].groupby("clf")["cost_s"].median().round(3)
display_status.to_csv(os.path.join(DIRS["tables"], "perfdb_status.csv"))
display_status

## 8. Lookup tables and dataset features
* **Lookup table** per (dataset, seed): validation/test balanced accuracy and cost for every classifier × budget. Cost is normalised by the cost of exhaustive search (all classifiers at 100%), so budget 1.0 = exhaustive.
* **17 dataset features** (computed on training rows only) go into the agent's state. There is no separate meta-learner: the RL policy learns from these directly.

In [ ]:
class Table:
    def __init__(self, g):
        self.val = np.full((K, L), np.nan); self.test = self.val.copy(); self.cost_s = self.val.copy()
        for r in g.itertuples(index=False):
            i, l = CLASSIFIERS.index(r.clf), int(r.level)
            self.val[i, l], self.test[i, l], self.cost_s[i, l] = r.val_bacc, r.test_bacc, max(r.cost_s, 1e-4)
        assert not np.isnan(self.val).any(), "incomplete performance data - rerun section 7"
        self.chance = float(g["chance"].iloc[0]); self.exhaustive_s = float(self.cost_s[:, -1].sum())
        self.cost = self.cost_s / self.exhaustive_s

TABLES = {(int(d), int(s)): Table(g) for (d, s), g in PERF.groupby(["did", "seed"])}
print(len(TABLES), "lookup tables")

STATE_FEATURES = ["log_n", "log_d", "log_d_encoded", "log_n_over_d", "n_classes", "cat_ratio", "missing_ratio",
                  "class_entropy_norm", "majority_ratio", "imbalance_ratio", "skew_mean", "kurt_mean",
                  "abs_corr_mean", "pca_frac_95", "pca_first_ratio", "mutual_info_mean", "mi_over_class_entropy"]

def _entropy(counts):
    p = np.asarray(counts, float); p = p[p > 0] / p.sum(); return float(-(p * np.log2(p)).sum())

def dataset_features(X, y, n_total):
    cats = categorical_columns(X); nums = [c for c in X.columns if c not in cats]
    Z = StandardPre().fit(X).transform(X)
    cnt = np.bincount(y); cnt = cnt[cnt > 0]; k = len(cnt)
    f = {"log_n": np.log10(n_total), "log_d": np.log10(X.shape[1]), "log_d_encoded": np.log10(max(1, Z.shape[1])),
         "log_n_over_d": np.log10(n_total / X.shape[1]), "n_classes": np.log(k), "cat_ratio": len(cats) / X.shape[1],
         "missing_ratio": float(X.isna().to_numpy().mean()), "class_entropy_norm": _entropy(cnt) / np.log2(k),
         "majority_ratio": cnt.max() / cnt.sum(), "imbalance_ratio": np.log(cnt.max() / cnt.min())}
    if nums:
        A = X[nums].to_numpy(float); A = np.where(np.isnan(A), np.nanmedian(A, axis=0), A); A = A[:, np.nanstd(A, axis=0) > 0]
        if A.shape[1]:
            f["skew_mean"] = float(np.nanmean(np.abs(stats.skew(A, axis=0))))
            f["kurt_mean"] = float(np.sign(m := np.nanmean(stats.kurtosis(A, axis=0))) * np.log1p(abs(m)))
            if A.shape[1] > 1:
                C = np.corrcoef(A[:, :50], rowvar=False); f["abs_corr_mean"] = float(np.nanmean(np.abs(C[np.triu_indices_from(C, 1)])))
    Zs = Z[:, :200]; Zs = Zs[:, Zs.std(axis=0) > 0]
    if Zs.shape[1]:
        p = PCA().fit(Zs); cum = np.cumsum(p.explained_variance_ratio_)
        f["pca_frac_95"] = float((np.searchsorted(cum, 0.95) + 1) / Zs.shape[1]); f["pca_first_ratio"] = float(p.explained_variance_ratio_[0])
        cols = np.random.default_rng(0).choice(Zs.shape[1], size=min(50, Zs.shape[1]), replace=False)
        mi = mutual_info_classif(Zs[:, cols], y, random_state=0)
        f["mutual_info_mean"] = float(mi.mean()); f["mi_over_class_entropy"] = float(mi.mean() / max(_entropy(cnt) * np.log(2), 1e-9))
    return f

feat_path = os.path.join(DIRS["features"], "dataset_features.csv")
FEAT = pd.read_csv(feat_path) if os.path.exists(feat_path) else pd.DataFrame(columns=["did", "seed"])
have = set(zip(FEAT["did"].astype(int), FEAT["seed"].astype(int))); new = []
for did in DIDS:
    d = None
    for seed in SEEDS:
        if (did, seed) in have: continue
        d = d or pd.read_pickle(os.path.join(DIRS["data"], f"{did}.pkl"))
        y_all = LabelEncoder().fit_transform(d["y"]); tr, _, _ = row_split(pd.Series(y_all), seed)
        pos = budget_subsample(y_all[tr], min(1.0, 2000 / len(tr)), seed)
        Xs, ys = d["X"].iloc[tr[pos]].reset_index(drop=True), LabelEncoder().fit_transform(y_all[tr[pos]])
        new.append({"did": did, "seed": seed, **dataset_features(Xs, ys, len(tr))})
if new:
    FEAT = pd.concat([FEAT, pd.DataFrame(new)], ignore_index=True); FEAT.to_csv(feat_path, index=False)
FEAT = FEAT.set_index(["did", "seed"]).reindex(columns=STATE_FEATURES).astype(float)
print("features:", FEAT.shape)

def build_context(train_dids):
    """Feature scaling + baseline statistics, fitted on the given TRAINING datasets only."""
    tr_keys = [k for k in TABLES if k[0] in set(train_dids)]
    Ftr = FEAT.loc[tr_keys]; med = Ftr.median(); filled = Ftr.fillna(med)
    mu, sd = filled.mean(), filled.std().replace(0, 1).fillna(1)
    ctx = {k: {"features": ((FEAT.loc[[k]].fillna(med) - mu) / sd).clip(-5, 5).fillna(0).to_numpy()[0]} for k in TABLES}
    perf = np.stack([TABLES[k].val[:, -1] for k in tr_keys])
    order = list(np.argsort((-perf).argsort(axis=1).argsort(axis=1).mean(axis=0)))
    avg_cost = np.stack([TABLES[k].cost[:, -1] for k in tr_keys]).mean(axis=0)
    return ctx, {"avg_rank_order": [int(i) for i in order], "avg_cost": avg_cost}

## 9. The RL environment (budget-aware selection MDP)
* **Actions** (K·L + 1): evaluate classifier *k* at budget level *l* (only levels above what *k* already reached: **Continue** = same classifier with more data, **Switch** = another classifier), or **STOP**.
* **State**: progress per classifier, best observed validation score per classifier (relative to best so far), observed cost per classifier (scaled to a 100% estimate), budget used/remaining/total, best score so far, 17 dataset features.
* **Reward**: −λ·cost per evaluation and 1 − regret at the end (training uses validation-based regret, so no test data shapes the policy). Training additionally uses potential-based shaping (Φ = normalised best score so far), which provably leaves the optimal policy unchanged (Ng, Harada & Russell, 1999), plus small noise on observed scores as a regulariser.
* **Affordability rule** (all methods): an evaluation whose *estimated* cost (average cost of that classifier on the training datasets × budget fraction) exceeds the remaining budget is not allowed. The true cost can still be higher; then the budget runs out mid-fit and no score is obtained.
* **Selection rule** (all methods): the classifier with the best observed validation score; its 100%-budget **test** score is reported. **Normalised regret** = (best − selected)/(best − worst) over the 12 classifiers.

In [ ]:
DEFAULT_FLAGS = {"obs": True, "cost": True, "budget": True, "features": True}

class Discrete:
    def __init__(self, n): self.n = n

class SelectionEnv:
    def __init__(self, ctx, stats_, instances, quality="val", train=False, flags=None, full_budget_only=False,
                 lam=ENV_CFG["lam"], gamma=0.99, seed=0):
        self.ctx, self.instances, self.quality, self.train = ctx, list(instances), quality, train
        self.flags = {**DEFAULT_FLAGS, **(flags or {})}; self.lam, self.gamma = lam, gamma
        self.cost_profile = np.asarray(stats_["avg_cost"]); self.rng = np.random.default_rng(seed)
        self.allowed = np.zeros(L, bool); self.allowed[[L - 1] if full_budget_only else list(range(L))] = True
        self.STOP = K * L; self.action_space = Discrete(K * L + 1)
        self.n_feat = len(STATE_FEATURES); self.obs_dim = 3 * K + 4 + self.n_feat
        self.obs_noise = ENV_CFG["obs_noise"] if train else 0.0
        self.shaping = ENV_CFG["shaping"] and train

    def action(self, k, l): return int(k) * L + int(l)
    def decode(self, a): return divmod(int(a), L)
    @property
    def remaining(self): return np.inf if self.budget is None else self.budget - self.used
    def predicted_cost(self, k, l): return float(self.cost_profile[k] * BUDGETS[l])
    def best_so_far(self):
        s = self.obs_score[~np.isnan(self.obs_score)]; return float(s.max()) if len(s) else np.nan

    def mask(self):
        m = np.zeros(K * L + 1, bool); lv = np.arange(L)
        afford = np.ones(L, bool)
        for k in range(K):
            if ENV_CFG.get("cost_mask") and self.budget is not None:
                afford = self.cost_profile[k] * np.asarray(BUDGETS) <= self.remaining + 1e-12
            m[k * L:(k + 1) * L] = (lv > self.level[k]) & self.allowed & afford
        m[self.STOP] = True; return m

    def _phi(self):
        b = self.best_so_far(); q = self.table.val[:, -1]
        if np.isnan(b) or q.max() <= q.min(): return 0.0
        return float(np.clip((b - q.min()) / (q.max() - q.min()), 0, 1))

    def reset(self, instance=None, budget="random"):
        self.key = tuple(instance) if instance is not None else self.instances[self.rng.integers(len(self.instances))]
        self.budget = float(self.rng.choice(BUDGETS)) if budget == "random" else budget
        self.table, self.c = TABLES[self.key], self.ctx[self.key]
        self.level = np.full(K, -1); self.obs_score = np.full(K, np.nan)
        self.used = self.raw_s = 0.0; self.n_evals = 0; self.trace = []; self.done = False
        return self.obs(), {"mask": self.mask()}

    def step(self, a):
        assert not self.done and self.mask()[a], "invalid action"
        phi0 = self._phi() if self.shaping else 0.0
        r, term, cut = 0.0, False, False
        if a == self.STOP:
            term = True
        else:
            k, l = self.decode(a); c = float(self.table.cost[k, l])
            if self.budget is not None and c > self.remaining + 1e-12:       # budget runs out mid-fit
                spent = self.budget - self.used; self.raw_s += spent * self.table.exhaustive_s
                self.used = self.budget; r -= self.lam * spent; term = cut = True
                self.trace.append({"clf": CLASSIFIERS[k], "budget": BUDGETS[l], "cutoff": True})
            else:
                self.used += c; self.raw_s += float(self.table.cost_s[k, l]); self.level[k] = l; self.n_evals += 1
                noise = self.rng.normal(0, self.obs_noise) if self.obs_noise > 0 else 0.0
                self.obs_score[k] = self.table.val[k, l] + noise; r -= self.lam * c
                self.trace.append({"clf": CLASSIFIERS[k], "budget": BUDGETS[l], "cutoff": False, "used": self.used,
                                   "val": float(self.table.val[k, l]), "best_val": self.best_so_far()})
                if not self.mask()[:self.STOP].any(): term = True
        info = {"mask": self.mask()}
        if self.shaping:
            r += (0.0 if term else self.gamma * self._phi()) - phi0
        if term:
            self.done = True; info.update(self.finish(), cutoff=cut)
            r += 1.0 - info["regret_val" if self.quality == "val" else "regret_test"]
        return self.obs(), float(r), term, info

    def _regret(self, q, s):
        if s is None: return 1.0
        span = q.max() - q.min(); return 0.0 if span <= 0 else float((q.max() - q[s]) / span)

    def finish(self):
        s = None if np.all(np.isnan(self.obs_score)) else int(np.nanargmax(self.obs_score))
        qv, qt = self.table.val[:, -1], self.table.test[:, -1]
        return {"selected": None if s is None else CLASSIFIERS[s], "true_best": CLASSIFIERS[int(np.argmax(qt))],
                "regret_val": self._regret(qv, s), "regret_test": self._regret(qt, s),
                "test_score": float(qt[s]) if s is not None else self.table.chance, "best_test": float(qt.max()),
                "budget_used": self.used, "raw_seconds": self.raw_s, "n_evals": self.n_evals, "trace": list(self.trace)}

    def obs(self):
        f = self.flags; parts = [(self.level + 1) / L]; best = self.best_so_far()
        parts.append(np.where(np.isnan(self.obs_score), 0, self.obs_score - (0 if np.isnan(best) else best)) if f["obs"] else np.zeros(K))
        seen = np.zeros(K)
        if f["cost"]:
            t = self.level >= 0; lv = self.level[t]; seen[t] = self.table.cost[t, lv] / np.asarray(BUDGETS)[lv]
        parts.append(np.clip(seen, 0, 2))
        B = 2.0 if self.budget is None else min(self.budget, 2.0)
        parts.append(np.array([min(self.used, 2), min(self.remaining, 2), B]) if f["budget"] else np.zeros(3))
        parts.append(np.array([0.0 if (np.isnan(best) or not f["obs"]) else best - self.table.chance]))
        parts.append(self.c["features"] if f["features"] else np.zeros(self.n_feat))
        return np.concatenate(parts).astype(np.float32)

    def discrete_state(self):
        B = 1.0 if self.budget is None else self.budget; tried = np.flatnonzero(self.level >= 0)
        best, blv = (int(np.nanargmax(self.obs_score)), int(self.level[np.nanargmax(self.obs_score)]) + 1) if len(tried) else (K, 0)
        b_idx = BUDGETS.index(self.budget) if self.budget in BUDGETS else len(BUDGETS)
        fz = self.c["features"]; bucket = (0 if fz[0] < -0.43 else 2 if fz[0] > 0.43 else 1) * 2 + int(fz[2] > 0)
        return (int(min(self.used / B, 0.999) * 4), best, blv, min(len(tried), 3), b_idx, bucket)

## 10. Agents: tabular Q-learning, DQN, Double DQN
The Q-network is a small MLP (2 × 128) written in NumPy with manual backpropagation and Adam: transparent and fast for this size (a GPU gives no speed-up for networks this small; the GPU is used for TabPFN/RealMLP above). DQN uses experience replay and a target network (Mnih et al., 2015); Double DQN decouples action selection and evaluation in the target (van Hasselt et al., 2016). Invalid actions are masked when acting and in the targets.

In [ ]:
NEG = -1e9

class MLP:
    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed); self.W, self.b = [], []
        for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
            s = np.sqrt(2.0 / a) if i < len(sizes) - 2 else 0.1 * np.sqrt(1.0 / a)
            self.W.append(rng.normal(0, s, (a, b))); self.b.append(np.zeros(b))
        self.t = 0; self.m = [np.zeros_like(p) for p in self.W + self.b]; self.v = [np.zeros_like(p) for p in self.W + self.b]
    def forward(self, x, cache=False):
        h, acts = np.asarray(x, float), []
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            acts.append(h); z = h @ W + b; h = np.maximum(z, 0) if i < len(self.W) - 1 else z
        return (h, acts) if cache else h
    def backward(self, acts, g):
        gW, gb = [None] * len(self.W), [None] * len(self.W)
        for i in reversed(range(len(self.W))):
            gW[i] = acts[i].T @ g; gb[i] = g.sum(0)
            if i: g = (g @ self.W[i].T) * (acts[i] > 0)
        return gW + gb
    def adam(self, grads, lr, clip=10.0):
        n = np.sqrt(sum((g ** 2).sum() for g in grads))
        if n > clip: grads = [g * clip / n for g in grads]
        self.t += 1; params = self.W + self.b
        for i, (p, g) in enumerate(zip(params, grads)):
            self.m[i] = 0.9 * self.m[i] + 0.1 * g; self.v[i] = 0.999 * self.v[i] + 0.001 * g * g
            p -= lr * (self.m[i] / (1 - 0.9 ** self.t)) / (np.sqrt(self.v[i] / (1 - 0.999 ** self.t)) + 1e-8)
    def get(self): return [w.copy() for w in self.W], [b.copy() for b in self.b]
    def set(self, p): self.W, self.b = [w.copy() for w in p[0]], [b.copy() for b in p[1]]
    def save(self, path): np.savez(path, **{f"W{i}": w for i, w in enumerate(self.W)}, **{f"b{i}": b for i, b in enumerate(self.b)})
    @classmethod
    def load(cls, path):
        z = np.load(path); n = len([k for k in z.files if k.startswith("W")]); m = cls.__new__(cls)
        m.W = [z[f"W{i}"] for i in range(n)]; m.b = [z[f"b{i}"] for i in range(n)]; return m

class DQNAgent:
    def __init__(self, obs_dim, n_act, cfg, double, seed=0):
        self.cfg, self.double, self.n_act = cfg, double, n_act; sizes = [obs_dim, *cfg["hidden"], n_act]
        self.q, self.qt = MLP(sizes, seed), MLP(sizes, seed); self.qt.set(self.q.get()); self.rng = np.random.default_rng(seed + 1)
        cap = cfg["buffer_size"]; self.cap, self.n, self.i = cap, 0, 0
        self.S = np.zeros((cap, obs_dim), np.float32); self.S2 = self.S.copy(); self.A = np.zeros(cap, int)
        self.R = np.zeros(cap, np.float32); self.D = np.zeros(cap, np.float32); self.M2 = np.zeros((cap, n_act), bool)
    def act(self, s, mask, eps=0.0):
        if eps and self.rng.random() < eps: return int(self.rng.choice(np.flatnonzero(mask)))
        return int(np.argmax(np.where(mask, self.q.forward(s[None])[0], NEG)))
    def store(self, s, a, r, s2, d, m2):
        j = self.i; self.S[j], self.A[j], self.R[j], self.S2[j], self.D[j], self.M2[j] = s, a, r, s2, d, m2
        self.i = (j + 1) % self.cap; self.n = min(self.n + 1, self.cap)
    def update(self):
        c = self.cfg; idx = self.rng.integers(0, self.n, c["batch_size"])
        s, a, r, s2, d, m2 = self.S[idx], self.A[idx], self.R[idx], self.S2[idx], self.D[idx], self.M2[idx]
        q, acts = self.q.forward(s, cache=True); qt2 = self.qt.forward(s2)
        if self.double:
            nxt = qt2[np.arange(len(a)), np.argmax(np.where(m2, self.q.forward(s2), NEG), 1)]
        else:
            nxt = np.where(m2, qt2, NEG).max(1)
        y = r + c["gamma"] * np.where(d > 0, 0.0, nxt); delta = q[np.arange(len(a)), a] - y
        g = np.zeros_like(q); g[np.arange(len(a)), a] = np.clip(delta, -1, 1) / len(a)
        self.q.adam(self.q.backward(acts, g), c["lr"], c["grad_clip"])
        ad = np.abs(delta)
        return float(np.mean(np.where(ad < 1, 0.5 * delta ** 2, ad - 0.5)))   # Huber loss

def train_dqn(env, agent, eval_fn):
    c = agent.cfg; decay = max(1, int(c["eps_decay_frac"] * c["train_steps"]))
    s, info = env.reset(); hist, rets, losses, G = [], [], [], 0.0; best, best_p = np.inf, agent.q.get()
    for t in range(1, c["train_steps"] + 1):
        eps = c["eps_end"] + (c["eps_start"] - c["eps_end"]) * max(0.0, 1 - t / decay)
        a = agent.act(s, info["mask"], eps); s2, r, done, info2 = env.step(a); G += r
        m2 = info2["mask"] if not done else np.eye(agent.n_act, dtype=bool)[-1]
        agent.store(s, a, r, s2, float(done), m2)
        if done: rets.append(G); G = 0.0; s, info = env.reset()
        else: s, info = s2, info2
        if t >= c["warmup"]: losses.append(agent.update())
        if t % c["target_update"] == 0: agent.qt.set(agent.q.get())
        if t % c["eval_every"] == 0 or t == c["train_steps"]:
            v = eval_fn(agent); hist.append({"step": t, "eps": eps, "train_return": np.mean(rets[-300:]) if rets else np.nan,
                                             "loss": np.mean(losses[-500:]) if losses else np.nan, "inner_val_regret": v})
            if v < best: best, best_p = v, agent.q.get()
    agent.q.set(best_p); return pd.DataFrame(hist)

class QAgent:
    def __init__(self, n_act, cfg, seed=0):
        self.n_act, self.cfg, self.Q, self.Qc, self.rng = n_act, cfg, {}, {}, np.random.default_rng(seed)
    def _g(self, T, s):
        if s not in T: T[s] = np.zeros(self.n_act)
        return T[s]
    def act(self, s, mask, eps=0.0):
        valid = np.flatnonzero(mask)
        if eps and self.rng.random() < eps: return int(self.rng.choice(valid))
        q = self.Q.get(s, self.Qc.get(s[:-1]))
        return int(valid[-1]) if q is None else int(np.argmax(np.where(mask, q, NEG)))
    def update(self, s, a, r, s2, m2, done):
        for T, x, x2 in ((self.Q, s, s2), (self.Qc, s[:-1], s2[:-1])):
            tgt = r if done else r + self.cfg["gamma"] * np.where(m2, self._g(T, x2), NEG).max()
            q = self._g(T, x); q[a] += self.cfg["alpha"] * (tgt - q[a])

def train_q(env, agent, eval_fn):
    c = agent.cfg; decay = max(1, int(c["eps_decay_frac"] * c["episodes"])); hist, rets = [], []; best, snap = np.inf, None
    for ep in range(1, c["episodes"] + 1):
        eps = c["eps_end"] + (c["eps_start"] - c["eps_end"]) * max(0.0, 1 - ep / decay)
        _, info = env.reset(); s, done, G = env.discrete_state(), False, 0.0
        while not done:
            a = agent.act(s, info["mask"], eps); _, r, done, info = env.step(a); s2 = env.discrete_state()
            agent.update(s, a, r, s2, info["mask"], done); s, G = s2, G + r
        rets.append(G)
        if ep % c["eval_every"] == 0 or ep == c["episodes"]:
            v = eval_fn(agent); hist.append({"step": ep, "eps": eps, "train_return": np.mean(rets[-500:]), "inner_val_regret": v})
            if v < best: best, snap = v, ({k: x.copy() for k, x in agent.Q.items()}, {k: x.copy() for k, x in agent.Qc.items()})
    if snap: agent.Q, agent.Qc = snap
    return pd.DataFrame(hist)

## 11. Baselines and policies
All baselines use the same environment, budget, selection rule and lookup data. Budget-aware baselines only use an **estimate** of cost (average cost of each classifier on the training datasets), never the true cost; screening rungs evaluate the **cheapest classifiers first** so expensive models (CatBoost, RealMLP, TabPFN) cannot use up a small budget before anything is observed.

In [ ]:
class Policy:
    budget_aware = True
    def reset(self, env): pass

def fits(env, k, l): return env.predicted_cost(k, l) <= env.remaining + 1e-12

class OrderedFull(Policy):
    def __init__(self, name, order_fn): self.name, self.order_fn = name, order_fn
    def reset(self, env): self.q = list(self.order_fn(env))
    def act(self, env, s, info):
        while self.q:
            k = self.q.pop(0); a = env.action(k, L - 1)
            if info["mask"][a] and fits(env, k, L - 1): return a
        return env.STOP

class OneShot(Policy):
    budget_aware = False
    def __init__(self, name, pick): self.name, self.pick = name, pick
    def reset(self, env): self.q = list(self.pick(env))
    def act(self, env, s, info):
        while self.q:
            a = env.action(self.q.pop(0), L - 1)
            if info["mask"][a]: return a
        return env.STOP

class BestOnSmall(Policy):
    name = "best_on_small"
    def reset(self, env): self.p1 = sorted(range(K), key=lambda k: env.cost_profile[k]); self.final = False
    def act(self, env, s, info):
        while self.p1:
            k = self.p1.pop(0)
            if fits(env, k, 0): return env.action(k, 0)
        if not self.final and not np.all(np.isnan(env.obs_score)):
            self.final = True; k = int(np.nanargmax(env.obs_score)); a = env.action(k, L - 1)
            if info["mask"][a] and fits(env, k, L - 1): return a
        return env.STOP

class SuccessiveHalving(Policy):
    name = "succ_halving"
    def reset(self, env): self.cands = list(range(K)); self.rung = 0; self.q = self._q(env)
    def _q(self, env): return sorted(self.cands, key=lambda k: env.cost_profile[k])
    def act(self, env, s, info):
        while True:
            if self.q:
                k = self.q.pop(0); a = env.action(k, self.rung)
                if not info["mask"][a]: continue
                if not fits(env, k, self.rung): self.cands.remove(k); continue
                return a
            self.cands = [k for k in self.cands if env.level[k] >= self.rung]
            if self.rung >= L - 1 or not self.cands: return env.STOP
            sc = env.obs_score[self.cands]; keep = max(1, math.ceil(len(self.cands) / 2))
            self.cands = [self.cands[i] for i in np.argsort(-np.nan_to_num(sc, nan=-1))[:keep]]
            self.rung += 1; self.q = self._q(env)

class AgentPolicy(Policy):
    def __init__(self, agent, name): self.agent, self.name = agent, name
    def act(self, env, s, info): return self.agent.act(s, info["mask"])

class QPolicy(Policy):
    def __init__(self, agent, name="qlearning"): self.agent, self.name = agent, name
    def act(self, env, s, info): return self.agent.act(env.discrete_state(), info["mask"])

def make_baselines(stats_, seed=0):
    rng = np.random.default_rng(seed); order = stats_["avg_rank_order"]
    pols = [OrderedFull("random", lambda env: rng.permutation(K)), OrderedFull("avg_rank", lambda env: order),
            OneShot("exhaustive", lambda env: range(K)), BestOnSmall(), SuccessiveHalving()]
    if "rf" in CLASSIFIERS: pols.append(OneShot("fixed_rf", lambda env: [CLASSIFIERS.index("rf")]))
    return pols

def run_episode(env, pol, key, B):
    s, info = env.reset(key, B); pol.reset(env)
    while True:
        s, r, done, info = env.step(pol.act(env, s, info))
        if done: return info

def evaluate(env, pols, keys, reps=None, extra=None):
    rows = []
    for p in pols:
        for key in keys:
            for B in (BUDGETS if p.budget_aware else [None]):
                for rep in range((reps or {}).get(p.name, 1)):
                    i = run_episode(env, p, key, B)
                    rows.append({"method": p.name, "did": key[0], "seed": key[1], "budget": np.nan if B is None else B,
                                 "rep": rep, "budget_aware": p.budget_aware, "selected": i["selected"], "true_best": i["true_best"],
                                 "regret": i["regret_test"], "test_bacc": i["test_score"], "best_test": i["best_test"],
                                 "budget_used": i["budget_used"], "seconds": i["raw_seconds"], "n_evals": i["n_evals"],
                                 "trace": json.dumps(i["trace"]), **(extra or {})})
    return pd.DataFrame(rows)

def mean_regret(env, pol, keys):
    return float(np.mean([run_episode(env, pol, k, B)["regret_val"] for k in keys for B in BUDGETS]))

## 12. Cross-validated training and evaluation
The 25 datasets are split into 5 folds (stratified by size and binary/multiclass). For each fold: the agents train on 16 datasets, pick their best checkpoint on 4 **inner-validation** datasets, and are tested on the 5 **unseen** datasets of that fold. Baseline statistics (average rank, average cost) and feature scaling also come from the training datasets of the fold only. Every dataset is therefore tested exactly once, by models that never saw it.

Saved per fold: agent weights (`.npz` for DQN/DDQN, `.pkl` for Q-learning), training histories and evaluation rows.

In [ ]:
def make_folds(n_folds, seed=42):
    m = META.set_index("did").loc[DIDS]
    key = pd.DataFrame({"size": pd.cut(m["n_rows"], [0, 2000, 10000, np.inf], labels=False).values,
                        "binary": (m["n_classes"] == 2).astype(int).values,
                        "r": np.random.default_rng(seed).random(len(m))}, index=m.index).sort_values(["size", "binary", "r"])
    return {int(d): i % n_folds for i, d in enumerate(key.index)}

FOLD_OF = make_folds(N_FOLDS)
save_json({str(k): v for k, v in FOLD_OF.items()}, os.path.join(DIRS["results"], "folds.json"))
print(pd.Series(FOLD_OF).value_counts().sort_index().rename("datasets per fold").to_string())

def fold_sets(f):
    test = [d for d in DIDS if FOLD_OF[d] == f]
    rest = [d for d in DIDS if FOLD_OF[d] != f]
    inner = make_folds_sub(rest, f)
    return [d for d in rest if d not in inner], inner, test

def make_folds_sub(dids, f, frac=0.2):
    m = META.set_index("did").loc[dids]
    key = pd.DataFrame({"size": pd.cut(m["n_rows"], [0, 2000, 10000, np.inf], labels=False).values,
                        "r": np.random.default_rng(100 + f).random(len(m))}, index=m.index).sort_values(["size", "r"])
    n = max(1, round(frac * len(dids))); step = len(dids) / n
    return [int(key.index[int(i * step)]) for i in range(n)]

def keys_of(dids): return sorted(k for k in TABLES if k[0] in set(dids))

ALGOS = ["qlearning", "dqn", "ddqn"]
all_rows, all_hist = [], []
t0 = time.time()
for f in range(N_FOLDS):
    tr, inner, te = fold_sets(f)
    fold_dir = os.path.join(DIRS["agents"], f"fold{f}"); os.makedirs(fold_dir, exist_ok=True)
    res_path = os.path.join(DIRS["results"], f"eval_fold{f}.csv")
    ctx, st = build_context(tr)
    env_train = lambda seed, gamma=0.99: SelectionEnv(ctx, st, keys_of(tr), "val", train=True, gamma=gamma, seed=seed)
    env_inner = SelectionEnv(ctx, st, keys_of(inner), "val")
    env_test = SelectionEnv(ctx, st, keys_of(te), "test")
    log(f"fold {f}: train {len(tr)} datasets, inner-val {len(inner)}, test {len(te)}: {te}")
    pols = make_baselines(st, seed=f)
    for algo in ALGOS:
        for seed in AGENT_SEEDS:
            wpath = os.path.join(fold_dir, f"{algo}_seed{seed}." + ("pkl" if algo == "qlearning" else "npz"))
            hpath = os.path.join(fold_dir, f"{algo}_seed{seed}_history.csv")
            if algo == "qlearning":
                ag = QAgent(K * L + 1, QL_CFG, seed)
                if os.path.exists(wpath):
                    ag.Q, ag.Qc = pd.read_pickle(wpath)
                else:
                    h = train_q(env_train(seed, gamma=QL_CFG["gamma"]), ag, lambda a: mean_regret(env_inner, QPolicy(a), keys_of(inner)))
                    pd.to_pickle((ag.Q, ag.Qc), wpath); h.to_csv(hpath, index=False)
                pol = QPolicy(ag, "qlearning")
            else:
                ag = DQNAgent(env_inner.obs_dim, K * L + 1, DQN_CFG, double=(algo == "ddqn"), seed=seed)
                if os.path.exists(wpath):
                    ag.q = MLP.load(wpath)
                else:
                    h = train_dqn(env_train(seed), ag, lambda a, n=algo: mean_regret(env_inner, AgentPolicy(a, n), keys_of(inner)))
                    ag.q.save(wpath); h.to_csv(hpath, index=False)
                pol = AgentPolicy(ag, algo)
            if os.path.exists(hpath):
                all_hist.append(pd.read_csv(hpath).assign(algo=algo, seed=seed, fold=f))
            pol.agent_seed = seed; pols.append(pol)
            log(f"  fold {f} {algo} seed {seed} ready ({time.time() - t0:.0f}s)")
    if os.path.exists(res_path):
        rows = pd.read_csv(res_path)
    else:
        parts = []
        for p in pols:
            df = evaluate(env_test, [p], keys_of(te), reps={"random": 10})
            if hasattr(p, "agent_seed"): df["rep"] = p.agent_seed
            parts.append(df)
        rows = pd.concat(parts, ignore_index=True).assign(fold=f); rows.to_csv(res_path, index=False)
    all_rows.append(rows)

RES = pd.concat(all_rows, ignore_index=True)
HIST = pd.concat(all_hist, ignore_index=True) if all_hist else pd.DataFrame()
RES.to_csv(os.path.join(DIRS["results"], "all_results.csv"), index=False)
HIST.to_csv(os.path.join(DIRS["results"], "training_histories.csv"), index=False)
log(f"cross-validation done: {RES['did'].nunique()} test datasets, {len(RES)} episodes")

## 13. Ablations (Double DQN, one seed per fold)
Each ablation removes one part of the agent to show what it contributes. The **one-shot** variant cannot see the scores it obtains, which tests the core claim that *sequential* decisions matter.

In [ ]:
ABL = pd.DataFrame()
if RUN_ABLATIONS:
    parts = []
    for f in range(N_FOLDS):
        tr, inner, te = fold_sets(f); ctx, st = build_context(tr)
        for name, spec in {"full": {}, **ABLATIONS}.items():
            out = os.path.join(DIRS["results"], f"ablation_{name}_fold{f}.csv")
            if os.path.exists(out):
                parts.append(pd.read_csv(out)); continue
            kw = {"full_budget_only": True} if spec == "full_budget_only" else {"flags": spec}
            ag = DQNAgent(3 * K + 4 + len(STATE_FEATURES), K * L + 1, DQN_CFG, double=True, seed=AGENT_SEEDS[0])
            if name == "full" and os.path.exists(os.path.join(DIRS["agents"], f"fold{f}", f"ddqn_seed{AGENT_SEEDS[0]}.npz")):
                ag.q = MLP.load(os.path.join(DIRS["agents"], f"fold{f}", f"ddqn_seed{AGENT_SEEDS[0]}.npz"))
            else:
                e_tr = SelectionEnv(ctx, st, keys_of(tr), "val", train=True, seed=AGENT_SEEDS[0], **kw)
                e_in = SelectionEnv(ctx, st, keys_of(inner), "val", **kw)
                train_dqn(e_tr, ag, lambda a: mean_regret(e_in, AgentPolicy(a, "ddqn"), keys_of(inner)))
                ag.q.save(os.path.join(DIRS["agents"], f"fold{f}", f"ablation_{name}.npz"))
            df = evaluate(SelectionEnv(ctx, st, keys_of(te), "test", **kw), [AgentPolicy(ag, name)], keys_of(te)).assign(fold=f, variant=name)
            df.to_csv(out, index=False); parts.append(df)
            log(f"  ablation {name} fold {f} done")
    ABL = pd.concat(parts, ignore_index=True); ABL.to_csv(os.path.join(DIRS["results"], "ablations.csv"), index=False)

## 14. Results tables and statistical tests
The dataset is the unit of analysis: seeds, agent seeds and repeats are averaged within each dataset first. 95% confidence intervals are bootstrap intervals over the datasets.

In [ ]:
RL = ["qlearning", "dqn", "ddqn"]
ORDER = ["ddqn", "dqn", "qlearning", "succ_halving", "best_on_small", "avg_rank", "random", "exhaustive", "fixed_rf"]
NICE = {"ddqn": "Double DQN", "dqn": "DQN", "qlearning": "Q-learning", "succ_halving": "Successive halving",
        "best_on_small": "Best-on-small-budget", "avg_rank": "Average rank", "random": "Random",
        "exhaustive": "Exhaustive (all at 100%)", "fixed_rf": "Always Random Forest"}

def per_dataset(r):
    r = r.copy(); r["budget"] = r["budget"].fillna(-1.0); r["hit"] = (r["regret"] <= 1e-12).astype(float)
    return r.groupby(["method", "did", "budget"], as_index=False)[["regret", "test_bacc", "budget_used", "seconds", "n_evals", "hit"]].mean()

def boot_ci(x, n=5000, seed=0):
    x = np.asarray(x, float); m = np.random.default_rng(seed).choice(x, (n, len(x))).mean(1)
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

G = per_dataset(RES)
rows = []
for (m, b), d in G.groupby(["method", "budget"]):
    lo, hi = boot_ci(d["regret"])
    rows.append({"method": m, "budget": "unlimited" if b < 0 else b, "n_datasets": len(d), "regret_mean": d.regret.mean(),
                 "ci_low": lo, "ci_high": hi, "regret_median": d.regret.median(), "test_bacc": d.test_bacc.mean(),
                 "budget_used": d.budget_used.mean(), "n_evals": d.n_evals.mean(), "top1_hit_rate": d.hit.mean()})
SUMMARY = pd.DataFrame(rows)
SUMMARY["order"] = SUMMARY["method"].map({m: i for i, m in enumerate(ORDER)})
SUMMARY = SUMMARY.sort_values(["budget", "regret_mean"], key=lambda s: s.astype(str) if s.name == "budget" else s).drop(columns="order")
SUMMARY.round(4).to_csv(os.path.join(DIRS["tables"], "summary.csv"), index=False)

trap = getattr(np, "trapezoid", None) or np.trapz
aw = G[G.budget >= 0]
AURC_PD = aw.groupby(["method", "did"]).apply(lambda d: trap(d.sort_values("budget").regret, d.sort_values("budget").budget) / (BUDGETS[-1] - BUDGETS[0])).rename("aurc").reset_index()
AURC = AURC_PD.groupby("method")["aurc"].agg(["mean", "median", "count"])
AURC[["ci_low", "ci_high"]] = [boot_ci(AURC_PD[AURC_PD.method == m].aurc) for m in AURC.index]
AURC = AURC.sort_values("mean"); AURC.round(4).to_csv(os.path.join(DIRS["tables"], "aurc.csv"))

# variability over agent seeds (RL only)
seed_var = (RES[RES.method.isin(RL) & RES.budget.notna()].groupby(["method", "rep", "did", "budget"])["regret"].mean().reset_index()
            .groupby(["method", "rep"])["regret"].mean().groupby("method").agg(["mean", "std"]))
seed_var.round(4).to_csv(os.path.join(DIRS["tables"], "rl_seed_variability.csv"))

def friedman_nemenyi(mat, alpha=0.05):
    k, n = mat.shape[1], mat.shape[0]
    st_, p = stats.friedmanchisquare(*[mat[c] for c in mat.columns])
    cd = stats.studentized_range.ppf(1 - alpha, k, np.inf) / np.sqrt(2) * np.sqrt(k * (k + 1) / (6 * n))
    return {"friedman_chi2": float(st_), "friedman_p": float(p), "nemenyi_cd": float(cd), "n_datasets": n, "n_methods": k}

def wilcoxon_holm(mat, ref="ddqn"):
    out = []
    for c in mat.columns:
        if c == ref: continue
        d = mat[ref] - mat[c]
        p = 1.0 if np.allclose(d, 0) else float(stats.wilcoxon(mat[ref], mat[c], zero_method="zsplit").pvalue)
        out.append({"vs": c, "mean_diff": d.mean(), "wins": int((d < -1e-12).sum()), "ties": int((abs(d) <= 1e-12).sum()),
                    "losses": int((d > 1e-12).sum()), "p": p})
    df = pd.DataFrame(out).sort_values("p").reset_index(drop=True); m = len(df); run = 0
    df["p_holm"] = [run := max(run, min(1, (m - i) * p)) for i, p in enumerate(df.p)]
    return df

def matrix_at(b):
    sel = G[(G.budget == b) | (G.budget < 0)]
    return sel.pivot_table(index="did", columns="method", values="regret").dropna()

TESTS, report = {}, []
for b in BUDGETS:
    mat = matrix_at(b); fr = friedman_nemenyi(mat); ranks = mat.rank(axis=1).mean().sort_values()
    wx = wilcoxon_holm(mat) if "ddqn" in mat else pd.DataFrame()
    TESTS[b] = {"mat": mat, "friedman": fr, "ranks": ranks, "wilcoxon": wx}
    wx.round(4).to_csv(os.path.join(DIRS["tables"], f"wilcoxon_ddqn_budget{b}.csv"), index=False)
    report += [f"\n### Budget {b}", f"Friedman chi2 = {fr['friedman_chi2']:.2f}, p = {fr['friedman_p']:.4f}; Nemenyi CD = {fr['nemenyi_cd']:.2f} (n = {fr['n_datasets']} datasets)",
               "Average ranks (1 = best):", ranks.round(2).to_string(), "\nDouble DQN vs each method (Wilcoxon signed-rank, Holm-corrected):", wx.round(4).to_string(index=False)]

# LaTeX + markdown table: mean regret per method x budget, best in bold
pv = SUMMARY.pivot_table(index="method", columns="budget", values="regret_mean", aggfunc="first")
pv = pv.reindex([m for m in ORDER if m in pv.index]); pv["AURC"] = AURC["mean"]
best = pv.min()
def fmt(v, col): return "" if pd.isna(v) else (f"\\textbf{{{v:.3f}}}" if np.isclose(v, best[col]) else f"{v:.3f}")
lines = ["\\begin{tabular}{l" + "c" * len(pv.columns) + "}", "\\toprule",
         "Method & " + " & ".join(str(c) for c in pv.columns) + " \\\\", "\\midrule"]
lines += [NICE.get(m, m) + " & " + " & ".join(fmt(pv.loc[m, c], c) for c in pv.columns) + " \\\\" for m in pv.index]
lines += ["\\bottomrule", "\\end{tabular}"]
open(os.path.join(DIRS["tables"], "regret_table.tex"), "w").write("\n".join(lines))
pv.rename(index=NICE).round(4).to_csv(os.path.join(DIRS["tables"], "regret_table.csv"))
open(os.path.join(DIRS["tables"], "statistical_tests.md"), "w").write("\n".join(report))

print("Mean normalised regret (lower is better), 25 unseen test datasets:")
pv.rename(index=NICE).round(4)

In [ ]:
print("Area under the regret-vs-budget curve (lower is better), with 95% CI:")
print(AURC.rename(index=NICE).round(4).to_string())
print("\nRL variability across agent seeds (mean regret per seed):")
print(seed_var.round(4).to_string())
print("\n".join(report))

## 15. Figures (saved to Drive as 200-dpi PNG)

In [ ]:
COL = {"ddqn": "#d62728", "dqn": "#ff7f0e", "qlearning": "#bcbd22", "succ_halving": "#1f77b4", "best_on_small": "#17becf",
       "avg_rank": "#9467bd", "random": "#7f7f7f", "exhaustive": "#2ca02c", "fixed_rf": "#8c564b"}
FIG = DIRS["figures"]
def savefig(fig, name):
    fig.tight_layout(); fig.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.show(); plt.close(fig)

# 1. regret vs budget
fig, ax = plt.subplots(figsize=(8, 4.8))
for m in [x for x in ORDER if x in set(aw.method)]:
    d = aw[aw.method == m].groupby("budget")["regret"]; mu = d.mean(); ci = np.array([boot_ci(v) for _, v in d])
    ax.plot(mu.index, mu.values, marker="o", lw=2.6 if m in RL else 1.4, color=COL[m], label=NICE[m])
    ax.fill_between(mu.index, ci[:, 0], ci[:, 1], color=COL[m], alpha=0.10 if m in RL else 0.05)
for m in ["exhaustive", "fixed_rf"]:
    d = G[(G.method == m)]
    if len(d): ax.scatter(d.budget_used.mean(), d.regret.mean(), marker="s", s=60, color=COL[m], zorder=5, label=NICE[m] + " (one-shot)")
ax.set_xscale("log"); ax.set_xticks(BUDGETS); ax.set_xticklabels([f"{int(b*100)}%" for b in BUDGETS])
ax.set_xlabel("budget (share of the cost of evaluating every classifier at 100%)"); ax.set_ylabel("normalised regret (lower is better)")
ax.set_title(f"Regret vs budget on {G.did.nunique()} unseen datasets (mean, 95% CI)", loc="left"); ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2, frameon=False); savefig(fig, "01_regret_vs_budget.png")

# 2. AURC bars
fig, ax = plt.subplots(figsize=(7.5, 4))
A = AURC.reindex([m for m in ORDER if m in AURC.index]).sort_values("mean")
ax.barh([NICE[m] for m in A.index], A["mean"], xerr=[A["mean"] - A.ci_low, A.ci_high - A["mean"]],
        color=[COL[m] for m in A.index], alpha=0.85, capsize=3)
for i, v in enumerate(A["mean"]): ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("area under regret-vs-budget curve (lower is better)")
ax.set_title("Overall budget-aware performance", loc="left"); savefig(fig, "02_aurc_bar.png")

# 3. critical-difference diagrams
def cd_plot(ranks, cd, title, name):
    names, r = list(ranks.index), ranks.values; k = len(r); o = np.argsort(r); sr = r[o]
    cl, last = [], -1
    for i in range(k):
        j = i
        while j + 1 < k and sr[j + 1] - sr[i] < cd: j += 1
        if j > i and j > last: cl.append((i, j)); last = j
    half = (k + 1) // 2; top = -0.35 - 0.18 * len(cl)
    fig, ax = plt.subplots(figsize=(8.5, 0.34 * half + 0.25 * len(cl) + 1.5))
    ax.set_xlim(0.5, k + 0.5); ax.set_ylim(top - 0.4 * half - 0.2, 0.9); ax.hlines(0, 1, k, color="k", lw=1)
    for i in range(1, k + 1): ax.vlines(i, -0.06, 0.06, color="k"); ax.text(i, 0.15, str(i), ha="center", fontsize=8)
    for c, (i, j) in enumerate(cl): ax.hlines(-0.25 - 0.18 * c, sr[i] - 0.05, sr[j] + 0.05, lw=3, color="tab:red", alpha=0.7)
    for j, idx in enumerate(o):
        left = j < half; y = top - 0.4 * (j if left else j - half); xt = 0.6 if left else k + 0.4
        ax.plot([r[idx], r[idx], xt], [0, y, y], color="0.4", lw=0.7)
        ax.text(xt, y, f" {NICE.get(names[idx], names[idx])} ({r[idx]:.2f}) ", ha="right" if left else "left", va="center", fontsize=8,
                fontweight="bold" if names[idx] in RL else "normal")
    ax.hlines(0.6, 1, min(1 + cd, k), lw=2, color="k"); ax.text(1 + min(cd, k - 1) / 2, 0.68, f"CD = {cd:.2f}", ha="center", fontsize=8)
    ax.axis("off"); ax.set_title(title, loc="left", fontsize=10); savefig(fig, name)

for b in BUDGETS:
    t = TESTS[b]
    cd_plot(t["ranks"], t["friedman"]["nemenyi_cd"],
            f"Average rank at budget {int(b*100)}% (Friedman p = {t['friedman']['friedman_p']:.3g}; bars join methods not significantly different)",
            f"03_cd_budget{int(b*100)}.png")

# 4. performance matrix: datasets x classifiers
P = PERF[PERF.level == L - 1].groupby(["did", "clf"])["test_bacc"].mean().unstack()[CLASSIFIERS]
P.index = [f"{d} {META.set_index('did').loc[d, 'name'][:18]}" for d in P.index]
fig, ax = plt.subplots(figsize=(1.0 + 0.62 * K, 0.33 * len(P) + 1.6))
im = ax.imshow(P.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(K), CLASSIFIERS, rotation=45, ha="right"); ax.set_yticks(range(len(P)), P.index, fontsize=7)
for i in range(P.shape[0]):
    j = int(np.nanargmax(P.values[i])); ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec="red", lw=1.5))
    for jj in range(K): ax.text(jj, i, f"{P.values[i, jj]:.2f}", ha="center", va="center", fontsize=5.5, color="w" if P.values[i, jj] < np.nanmean(P.values) else "k")
fig.colorbar(im, ax=ax, label="test balanced accuracy (100% budget)", shrink=0.6)
ax.set_title("No single classifier wins everywhere (red box = best per dataset)", loc="left"); savefig(fig, "04_performance_matrix.png")

# 5. how often each classifier is the best
wins = PERF[PERF.level == L - 1].groupby(["did", "seed"]).apply(lambda d: d.loc[d.test_bacc.idxmax(), "clf"]).value_counts().reindex(CLASSIFIERS).fillna(0)
fig, ax = plt.subplots(figsize=(7, 3.4)); ax.bar(wins.index, wins.values, color="#1f77b4")
ax.set_ylabel("times best (dataset x seed)"); ax.set_title("Which classifier is best varies across datasets", loc="left")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right"); savefig(fig, "05_best_classifier_counts.png")

# 6. pairwise win-rate matrix (AURC per dataset)
W = AURC_PD.pivot(index="did", columns="method", values="aurc"); ms = [m for m in ORDER if m in W.columns]
WR = pd.DataFrame([[np.mean(W[a] < W[b] - 1e-12) if a != b else np.nan for b in ms] for a in ms], index=ms, columns=ms)
fig, ax = plt.subplots(figsize=(6.8, 5.6)); im = ax.imshow(WR.values, cmap="RdBu", vmin=0, vmax=1)
ax.set_xticks(range(len(ms)), [NICE[m] for m in ms], rotation=45, ha="right", fontsize=8); ax.set_yticks(range(len(ms)), [NICE[m] for m in ms], fontsize=8)
for i in range(len(ms)):
    for j in range(len(ms)):
        if i != j: ax.text(j, i, f"{WR.values[i, j]:.0%}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="share of datasets where row beats column", shrink=0.8)
ax.set_title("Pairwise win rate (by area under regret curve)", loc="left"); savefig(fig, "06_win_rate_matrix.png")

# 7. selection confusion matrix (Double DQN, budget 25%)
for b in [0.25, 1.0]:
    S = RES[(RES.method == "ddqn") & (RES.budget == b)]
    if len(S):
        CM = pd.crosstab(S.true_best, S.selected.fillna("none")).reindex(index=CLASSIFIERS, columns=CLASSIFIERS + ["none"], fill_value=0)
        CM = CM.loc[CM.sum(1) > 0]
        fig, ax = plt.subplots(figsize=(0.55 * (K + 1) + 2, 0.45 * len(CM) + 1.8))
        im = ax.imshow(CM.values, cmap="Blues", aspect="auto")
        ax.set_xticks(range(CM.shape[1]), CM.columns, rotation=45, ha="right", fontsize=8); ax.set_yticks(range(len(CM)), CM.index, fontsize=8)
        for i in range(CM.shape[0]):
            for j in range(CM.shape[1]):
                if CM.values[i, j]: ax.text(j, i, int(CM.values[i, j]), ha="center", va="center", fontsize=7, color="w" if CM.values[i, j] > CM.values.max() / 2 else "k")
        ax.set_xlabel("classifier selected by Double DQN"); ax.set_ylabel("true best classifier (test)")
        ax.set_title(f"Selection matrix, budget {int(b*100)}% (diagonal = picked the true best)", loc="left"); savefig(fig, f"07_selection_matrix_ddqn_{int(b*100)}.png")

# 8. what the agent evaluates
for m in ["ddqn", "succ_halving"]:
    d = RES[(RES.method == m) & RES.budget.notna()]; M = np.zeros((K, L))
    for tr_ in d.trace:
        for s_ in json.loads(tr_):
            if not s_["cutoff"]: M[CLASSIFIERS.index(s_["clf"]), BUDGETS.index(s_["budget"])] += 1
    M /= max(1, len(d))
    fig, ax = plt.subplots(figsize=(5.2, 0.34 * K + 1.6)); im = ax.imshow(M, cmap="Blues", aspect="auto")
    ax.set_yticks(range(K), CLASSIFIERS, fontsize=8); ax.set_xticks(range(L), [f"{int(b*100)}%" for b in BUDGETS])
    for i in range(K):
        for j in range(L): ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7, color="w" if M[i, j] > M.max() / 2 else "k")
    fig.colorbar(im, ax=ax, label="evaluations per episode"); ax.set_xlabel("budget level")
    ax.set_title(f"What {NICE[m]} evaluates", loc="left"); savefig(fig, f"08_actions_{m}.png")

# 9. training curves
if len(HIST):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for algo in RL:
        h = HIST[HIST.algo == algo]
        if not len(h): continue
        h = h.assign(frac=h.groupby(["fold", "seed"])["step"].transform(lambda s: s / s.max()).round(2))
        for ax, col in zip(axes, ["inner_val_regret", "train_return"]):
            g = h.groupby("frac")[col]; mu, sd = g.mean(), g.std().fillna(0)
            ax.plot(mu.index, mu.values, color=COL[algo], lw=2, label=NICE[algo]); ax.fill_between(mu.index, mu - sd, mu + sd, color=COL[algo], alpha=0.15)
    axes[0].set_ylabel("inner-validation regret"); axes[1].set_ylabel("training return (episode)")
    for ax in axes: ax.set_xlabel("fraction of training"); ax.grid(alpha=0.3); ax.legend(frameon=False, fontsize=8)
    axes[0].set_title("Learning curves (mean ± sd over folds and seeds)", loc="left"); savefig(fig, "09_training_curves.png")

# 10. cost vs regret
s_ = G.groupby("method")[["budget_used", "regret"]].mean()
fig, ax = plt.subplots(figsize=(6.5, 4.4))
for m, r in s_.iterrows():
    ax.scatter(r.budget_used, r.regret, s=80 if m in RL else 40, color=COL.get(m, "k")); ax.annotate(NICE.get(m, m), (r.budget_used, r.regret), fontsize=8, xytext=(5, 3), textcoords="offset points")
ax.set_xlabel("mean budget actually used"); ax.set_ylabel("mean normalised regret"); ax.grid(alpha=0.3)
ax.set_title("Cost-quality trade-off (bottom-left is best)", loc="left"); savefig(fig, "10_cost_vs_regret.png")

# 11. ablations
if len(ABL):
    Ga = per_dataset(ABL.assign(method=ABL["variant"]))
    Aa = Ga[Ga.budget >= 0].groupby(["method", "did"]).apply(lambda d: trap(d.sort_values("budget").regret, d.sort_values("budget").budget) / (BUDGETS[-1] - BUDGETS[0])).rename("aurc").reset_index()
    tab = Aa.groupby("method")["aurc"].agg(["mean"]); tab[["ci_low", "ci_high"]] = [boot_ci(Aa[Aa.method == m].aurc) for m in tab.index]
    tab = tab.sort_values("mean"); tab.round(4).to_csv(os.path.join(DIRS["tables"], "ablations_aurc.csv"))
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.barh(tab.index, tab["mean"], xerr=[tab["mean"] - tab.ci_low, tab.ci_high - tab["mean"]], capsize=3,
            color=["#d62728" if i == "full" else "#999999" for i in tab.index])
    for i, v in enumerate(tab["mean"]): ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=8)
    ax.invert_yaxis(); ax.set_xlabel("area under regret curve (lower is better)")
    ax.set_title("Ablations of Double DQN (red = full agent)", loc="left"); savefig(fig, "11_ablations.png")
    print(tab.round(4))

# 12. search traces on three datasets (budget 25%)
for did in sorted(RES.did.unique())[:3]:
    fig, ax = plt.subplots(figsize=(6.8, 3.8))
    for m in ["ddqn", "succ_halving", "best_on_small", "random"]:
        d = RES[(RES.method == m) & (RES.did == did) & (RES.budget == 0.25) & (RES.seed == SEEDS[0])]
        if not len(d): continue
        steps = [s_ for s_ in json.loads(d.iloc[0].trace) if not s_["cutoff"]]
        if not steps: continue
        ax.step([0] + [s_["used"] for s_ in steps], [np.nan] + [s_["best_val"] for s_ in steps], where="post", color=COL[m], lw=2 if m in RL else 1.2, label=NICE[m])
    ax.set_xlabel("budget used"); ax.set_ylabel("best validation score so far"); ax.grid(alpha=0.3); ax.legend(fontsize=8, frameon=False)
    ax.set_title(f"Search trace: {META.set_index('did').loc[did, 'name']} (budget 25%)", loc="left"); savefig(fig, f"12_trace_{did}.png")

## 16. Summary file and download
Writes `SUMMARY.md` (numbers computed from your run) and a zip of the whole run folder.

In [ ]:
best_overall = AURC.index[0]
lines = [f"# AutoClassify Stage 2 — {RUN_NAME}", "",
         f"* datasets: {len(DIDS)} OpenML datasets, {N_FOLDS}-fold cross-validation over datasets (each tested once, unseen)",
         f"* classifiers ({K}): {', '.join(CLASSIFIERS)}", f"* budgets: {BUDGETS}; row seeds: {SEEDS}; agent seeds: {AGENT_SEEDS}",
         f"* hardware: {HARDWARE}", "", "## Area under regret-vs-budget curve (lower is better)", "",
         AURC.rename(index=NICE).round(4).to_string(), "",
         f"Best method overall by AURC: {NICE.get(best_overall, best_overall)}.", "",
         "## Mean regret per budget", "", pv.rename(index=NICE).round(4).to_string(), "",
         "## Statistical tests", *report, "", "Figures: figures/ | Tables: tables/ | Weights: agents/fold*/ | Raw results: results/"]
open(os.path.join(OUT, "SUMMARY.md"), "w").write("\n".join(lines))

import zipfile
zip_path = os.path.join(DRIVE_ROOT, f"{RUN_NAME}_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:   # results only (no raw data / OpenML cache)
    for sub in ["figures", "tables", "results", "agents", "features", "perfdb"]:
        for root, _, files in os.walk(DIRS[sub]):
            for fn in files:
                full = os.path.join(root, fn); z.write(full, os.path.relpath(full, OUT))
    for fn in ["SUMMARY.md", "run_config.json"]:
        z.write(os.path.join(OUT, fn), fn)
print("Saved in Drive:", OUT)
print("Zip of results, figures, tables and weights:", zip_path)
print("\n".join(lines[:14]))